# Antenna VAE Training

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import wandb
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from pathlib import Path

In [54]:
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [3]:
class AntennaDataset(Dataset):
    def __init__(self, design_params, s11_values):
        """
        Args:
            design_params: Array of shape (n_samples, 3) containing [length, width, feed_pos]
            s11_values: Array of shape (n_samples, n_freq_points) containing S11 values
        """
        self.design_scaler = StandardScaler()
        self.s11_scaler = StandardScaler()
        
        self.design_params = torch.FloatTensor(
            self.design_scaler.fit_transform(design_params)
        )
        
        self.s11_values = torch.FloatTensor(
            self.s11_scaler.fit_transform(s11_values)
        )
        
    def __len__(self):
        return len(self.design_params)
    
    def __getitem__(self, idx):
        return self.design_params[idx], self.s11_values[idx]
    
    def inverse_transform_design(self, design_params):
        """Convert normalized design parameters back to original scale"""
        return self.design_scaler.inverse_transform(design_params)
    
    def inverse_transform_s11(self, s11):
        """Convert normalized S11 values back to original scale"""
        return self.s11_scaler.inverse_transform(s11)

In [4]:
from pytorch_tcn import TCN
from typing import List, Optional
import torch.nn as nn


class S11Encoder(nn.Module):
    def __init__(
        self, latent_dim: int = 32, depth: int = 8, channels: Optional[List[int]] = None
    ):
        super(S11Encoder, self).__init__()

        if channels is None:
            channels = [latent_dim] * depth

        self.encoder = nn.Sequential(
            TCN(
                num_inputs=1,  
                num_channels=channels,  
                kernel_size=4,  
                dilations=None,  
                dilation_reset=None,  
                dropout=0.1,  
                causal=False,  
                use_norm="weight_norm",  
                activation="relu",  
                kernel_initializer="xavier_uniform",  
                use_skip_connections=True,  
                input_shape="NCL",  
                embedding_shapes=None,  
                use_gate=False,  
                output_projection=None,  
                output_activation=None,  
            ),
            nn.AdaptiveAvgPool1d(
                1
            ),  # Pool over the sequence length
            nn.Flatten(),  # Flatten the output to (batch_size, latent_dim)
        )

    def forward(self, x):
        # x is of shape (batch_size, 1000, 1)
        x = x.transpose(1, 2)  # Shape: (batch_size, 1, 1000)
        x = self.encoder(x)
        return x  # Output shape: (batch_size, latent_dim)

class S11Decoder(nn.Module):
    def __init__(self, latent_dim, output_length=1000, output_channels=1):
        super(S11Decoder, self).__init__()
        self.latent_dim = latent_dim
        self.output_length = output_length
        self.output_channels = output_channels

        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(
                in_channels=latent_dim,
                out_channels=512,
                kernel_size=4,
                stride=4,
                padding=0
            ),  # Output: (batch_size, 512, 4)
            nn.ReLU(),

            nn.ConvTranspose1d(
                in_channels=512,
                out_channels=256,
                kernel_size=5,
                stride=5,
                padding=0
            ),  # Output: (batch_size, 256, 20)
            nn.ReLU(),

            nn.ConvTranspose1d(
                in_channels=256,
                out_channels=128,
                kernel_size=5,
                stride=5,
                padding=0
            ),  # Output: (batch_size, 128, 100)
            nn.ReLU(),

            nn.ConvTranspose1d(
                in_channels=128,
                out_channels=64,
                kernel_size=10,
                stride=10,
                padding=0
            ),  # Output: (batch_size, 64, 1000)
            nn.ReLU(),

            nn.Conv1d(
                in_channels=64,
                out_channels=self.output_channels,
                kernel_size=1,
                padding=0
            ),  # Output: (batch_size, 1, 1000)
        )

    def forward(self, x):
        # x is of shape (batch_size, latent_dim)
        x = x.unsqueeze(-1)  # Reshape to (batch_size, latent_dim, 1)
        x = self.decoder(x)
        return x  # Output shape: (batch_size, output_channels, output_length)


class LatentSpace(nn.Module):
    def __init__(self, hidden_dim=64, latent_dim=8):
        super().__init__()
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_var = nn.Linear(hidden_dim, latent_dim)
        
    def forward(self, x):
        mu = self.fc_mu(x)
        log_var = self.fc_var(x)
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std


class DesignDecoder(nn.Module):
    def __init__(self, latent_dim=8, hidden_dim=64):
        super().__init__()
        
        self.layers = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, 3)
        )
        
    def forward(self, z):
        return self.layers(z)

class AntennaVAE(nn.Module):
    def __init__(self, n_freq_points=1000, latent_dim=8, hidden_dim=64):
        super().__init__()
        
        self.encoder = S11Encoder(latent_dim=hidden_dim)
        self.latent = LatentSpace(hidden_dim, latent_dim)
        self.s11_decoder = S11Decoder(latent_dim=latent_dim, output_length=n_freq_points)
        self.design_decoder = DesignDecoder(latent_dim, hidden_dim)
        
    def encode(self, s11_values):
        hidden = self.encoder(s11_values)
        return self.latent(hidden)
    
    def decode(self, z):
        design_params = self.design_decoder(z)
        s11_values = self.s11_decoder(z)
        return design_params, s11_values
    
    def forward(self, design_params, s11_values):
        mu, log_var = self.encode(s11_values)
        z = self.latent.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var
        
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

In [79]:
def weighted_s11_loss(recon_s11, s11_values):
    """
    Weight the loss more heavily in regions where S11 is lower (resonances)
    with proper normalization
    """
    # Create weights based on S11 values (higher weight for resonances)
    # Normalize S11 values to prevent exponential overflow
    normalized_s11 = (s11_values - torch.mean(s11_values)) / torch.std(s11_values)
    weights = torch.exp(-normalized_s11)
    
    # Normalize weights to sum to 1
    weights = weights / weights.sum(dim=1, keepdim=True)
    
    squared_diff = (recon_s11 - s11_values)**2
    weighted_loss = weights * squared_diff
    
    return torch.mean(weighted_loss)

def smoothness_loss(s11_values):
    """
    Penalize non-smooth variations in S11 response
    """
    return torch.mean(torch.abs(torch.diff(s11_values, dim=1)))


def vae_loss(recon_design, recon_s11, design_params, s11_values, mu, log_var, 
             design_weight=1.0, s11_weight=1.0, kl_weight=0.1, 
             smoothness_weight=0.1, weight_resonances=True):
    """Custom loss function combining reconstruction and KL divergence losses"""
    # Design parameters reconstruction loss
    design_loss = F.mse_loss(recon_design, design_params, reduction='mean')
    
    # S11 reconstruction loss with resonance weighting
    if weight_resonances:
        s11_loss = weighted_s11_loss(recon_s11, s11_values)
    else:
        s11_loss = F.mse_loss(recon_s11, s11_values, reduction='mean')
    
    # Smoothness loss for S11 curves
    smooth_loss = smoothness_loss(recon_s11)
    
    # KL divergence
    kl_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
    
    total_loss = (design_weight * design_loss + 
                 s11_weight * s11_loss + 
                 kl_weight * kl_loss +
                 smoothness_weight * smooth_loss)
    
    return total_loss, design_loss, s11_loss, kl_loss, smooth_loss

def train_epoch(model, train_loader, optimizer, device, epoch, config):
    model.train()
    total_loss = 0
    
    for batch_idx, (design_params, s11_values) in enumerate(train_loader):
        design_params = design_params.to(device)
        s11_values = s11_values.to(device)
        
        optimizer.zero_grad()
        (recon_design, recon_s11), mu, log_var = model(design_params, s11_values)
        
        loss, design_loss, s11_loss, kl_loss, smooth_loss = vae_loss(
            recon_design, recon_s11, design_params, s11_values, mu, log_var, weight_resonances=config.weight_resonances
        )
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Log batch metrics
        wandb.log({
            'batch_loss': loss.item(),
            'batch_design_loss': design_loss.item(),
            'batch_s11_loss': s11_loss.item(),
            'batch_kl_loss': kl_loss.item(),
            'batch_smooth_loss': smooth_loss.item()
        })
    
    avg_loss = total_loss / len(train_loader)
    return avg_loss

def validate(model, val_loader, device, config):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for design_params, s11_values in val_loader:
            design_params = design_params.to(device)
            s11_values = s11_values.to(device)
            
            (recon_design, recon_s11), mu, log_var = model(design_params, s11_values)
            loss, _, _, _, _ = vae_loss(
                recon_design, recon_s11, design_params, s11_values, mu, log_var, weight_resonances=config.weight_resonances
            )
            total_loss += loss.item()
    
    return total_loss / len(val_loader)

In [80]:
def plot_s11_comparison(model, dataset, device, n_samples=5):
    """Plot comparison of original vs reconstructed S11 curves"""
    model.eval()
    fig, axes = plt.subplots(n_samples, 1, figsize=(10, 3*n_samples))
    
    indices = np.random.choice(len(dataset), n_samples, replace=False)
    freq_points = np.linspace(0, 1, dataset.s11_values.shape[1])  # Normalized frequency points
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            design_params, s11_values = dataset[idx]
            design_params = design_params.unsqueeze(0).to(device)
            s11_values = s11_values.unsqueeze(0).to(device)
            
            (recon_design, recon_s11), _, _ = model(design_params, s11_values)
            
            # Convert to numpy and denormalize
            original_s11 = dataset.inverse_transform_s11(s11_values.cpu().numpy())[0]
            recon_s11 = dataset.inverse_transform_s11(recon_s11.cpu().numpy())[0]
            
            axes[i].plot(freq_points, original_s11, label='Original')
            axes[i].plot(freq_points, recon_s11, '--', label='Reconstructed')
            axes[i].set_ylabel('S11 (dB)')
            axes[i].legend()
    
    axes[-1].set_xlabel('Normalized Frequency')
    plt.tight_layout()
    return fig

In [81]:
def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')

In [82]:
def train_vae(design_params, s11_values, config):
    """Main training function"""
    # Initialize wandb
    wandb.init(project="antenna-vae", config=config)
    
    # Set device
    device = get_device()
    
    # Create dataset and dataloaders
    dataset = AntennaDataset(design_params, s11_values)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size]
    )
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config.batch_size, 
        shuffle=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=config.batch_size, 
        shuffle=False
    )
    
    # Initialize model
    model = AntennaVAE(
        n_freq_points=s11_values.shape[1],
        latent_dim=config.latent_dim,
        hidden_dim=config.hidden_dim
    ).to(device)
    
    # Initialize optimizer
    optimizer = torch.optim.Adam(
        model.parameters(), 
        lr=config.learning_rate
    )
    
    # Training loop
    best_val_loss = float('inf')
    for epoch in range(config.epochs):
        train_loss = train_epoch(
            model, train_loader, optimizer, device, epoch, config
        )
        val_loss = validate(model, val_loader, device, config)
        
        # Plot and log S11 comparisons periodically
        if epoch % 10 == 0:
            fig = plot_s11_comparison(model, dataset, device)
            wandb.log({"s11_comparison": wandb.Image(fig)})
            plt.close(fig)
        
        # Log metrics
        wandb.log({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss
        })
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 
                      Path(wandb.run.dir) / 'best_model.pt')
        
        print(f'Epoch {epoch}: Train Loss = {train_loss:.4f}, '
              f'Val Loss = {val_loss:.4f}')
    
    wandb.finish()
    return model, dataset

In [84]:
# Configuration
from types import SimpleNamespace   
config = {
    'batch_size': 32,
    'epochs': 100,
    'learning_rate': 1e-3,
    'latent_dim': 8,
    'hidden_dim': 64,
    'design_weight': 1.0,
    's11_weight': 1.0,
    'kl_weight': 0.5,
    'smoothness_weight': 0,
    'weight_resonances': False
}
config = SimpleNamespace(**config)
# Create an object from the dict

data_dir = "../data/results/sim_results2/preprocessed/"
design_params = np.load(data_dir + "design_params.npy")
freq_response = np.load(data_dir + "freq_response.npy")
s11_values = freq_response[:, :, 1]  # Extract just S11 values

# design_params: shape (555, 3)
# s11_values: shape (555, 1000)

# Train model
model, dataset = train_vae(design_params, s11_values, config)

Epoch 0: Train Loss = 1.9763, Val Loss = 2.2021
Epoch 1: Train Loss = 1.6375, Val Loss = 1.7111
Epoch 2: Train Loss = 1.5099, Val Loss = 1.4234
Epoch 3: Train Loss = 1.4075, Val Loss = 1.3813
Epoch 4: Train Loss = 1.3342, Val Loss = 1.3073
Epoch 5: Train Loss = 1.2771, Val Loss = 1.2822
Epoch 6: Train Loss = 1.2651, Val Loss = 1.2206
Epoch 7: Train Loss = 1.2037, Val Loss = 1.1827
Epoch 8: Train Loss = 1.1650, Val Loss = 1.1983
Epoch 9: Train Loss = 1.1573, Val Loss = 1.1597
Epoch 10: Train Loss = 1.1015, Val Loss = 1.1110
Epoch 11: Train Loss = 1.0757, Val Loss = 1.1170
Epoch 12: Train Loss = 1.0620, Val Loss = 1.1040
Epoch 13: Train Loss = 1.0142, Val Loss = 1.0337
Epoch 14: Train Loss = 1.0109, Val Loss = 1.0498
Epoch 15: Train Loss = 0.9685, Val Loss = 0.9802
Epoch 16: Train Loss = 0.9444, Val Loss = 0.9788
Epoch 17: Train Loss = 0.8916, Val Loss = 0.9500
Epoch 18: Train Loss = 0.8855, Val Loss = 0.9452
Epoch 19: Train Loss = 0.8735, Val Loss = 0.9209
Epoch 20: Train Loss = 0.8291,

batch_design_loss,▆▆█▅▅▄▄▅▄▅▂▄▃▂▂▂▁▂▂▂▂▃▂▂▂▂▁▃▂▂▂▁▂▁▁▂▂▁▂▁
batch_kl_loss,▁▁▂▂▂▃▃▆▇▇█▇█▇█▇▇▇▇██▇▇█▇█▇▇▇▇▇█▇▇██▇█▇▇
batch_loss,█▆▆▅▄▄▄▄▃▃▃▃▃▂▃▂▂▂▂▂▁▂▂▁▂▁▂▂▁▂▁▁▁▁▁▂▁▁▁▂
batch_s11_loss,██▇▇█▇▅▅▆▆▄▅▃▄▃▂▂▃▂▃▂▂▂▂▂▂▁▂▁▂▁▁▁▂▁▂▂▁▁▁
batch_smooth_loss,█▆▅▃▁▁▁▁▁▁▁▁▁▁▁▁▂▂▁▂▁▂▂▂▁▂▂▂▂▂▂▂▂▂▁▂▂▂▂▂
epoch,▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇████
train_loss,█▆▆▅▅▅▄▄▄▄▄▃▂▂▂▂▂▂▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch_design_loss,0.10163
batch_kl_loss,1.6332
batch_loss,0.40109
